# Algoritmo Genético para Maximizar una Función de Dos Variables

Queremos maximizar la función: 

$z = \frac{4x^2 - 4y^2}{3}$

con $x, y \in [-32, 31]$. En este cuaderno lo programamos paso por paso con codificación binaria para que puedas entender cada parte del algoritmo genético.

## Definir parámetros

Cada individuo tendrá bits para representar $x$ y $y$. Mientras más bits se use, mejor aproximidad de valores reales del intervalo.

In [4]:
import random

random.seed(7)

BITS_POR_VARIABLE = 10
LONGITUD_CROMOSOMA = BITS_POR_VARIABLE * 2
LIM_INF = -32
LIM_SUP = 31
TAM_POBLACION = 40
GENERACIONES = 80
PROB_CRUCE = 0.8
PROB_MUTACION = 1 / LONGITUD_CROMOSOMA

print(f'Bits por variable: {BITS_POR_VARIABLE}')
print(f'Longitud del cromosoma: {LONGITUD_CROMOSOMA}')
print(f'Probabilidad de mutación: {PROB_MUTACION:.4f}')

Bits por variable: 10
Longitud del cromosoma: 20
Probabilidad de mutación: 0.0500


## Representar un individuo

La primera mitad del cromosoma representará $x$ y la segunda mitad representará $y$. Luego convertimos cada bloque binario a un valor real dentro de $[-32, 31]$.

In [3]:
def generar_cromosoma():
    return ''.join(random.choice('01') for _ in range(LONGITUD_CROMOSOMA))

def binario_a_real(bits):
    entero = int(bits, 2)
    max_entero = (2 ** len(bits)) - 1
    return LIM_INF + entero * (LIM_SUP - LIM_INF) / max_entero

def decodificar(cromosoma):
    bits_x = cromosoma[:BITS_POR_VARIABLE]
    bits_y = cromosoma[BITS_POR_VARIABLE:]
    x = binario_a_real(bits_x)
    y = binario_a_real(bits_y)
    return x, y

ejemplo = generar_cromosoma()
print('Cromosoma de ejemplo:', ejemplo)
print('Valores decodificados:', decodificar(ejemplo))

Cromosoma de ejemplo: 10100010000110001000
Valores decodificados: (7.906158357771261, -7.859237536656892)


## Definir la función fitness

Como el objetivo es maximizar $z$, podemos usar la misma función como medida de aptitud.

In [6]:
def fitness(cromosoma):
    x, y = decodificar(cromosoma)
    return (4 * x**2 - 4 * y**2) / 3

print('Tabla de fitness para 5 cromosomas de ejemplo:')
for _ in range(5):
    cromosoma = generar_cromosoma()
    print(f'Cromosoma: {cromosoma}, Fitness: {fitness(cromosoma):.4f}') 

Tabla de fitness para 5 cromosomas de ejemplo:
Cromosoma: 10100010000110001000, Fitness: 0.9863
Cromosoma: 01000011001000100001, Fitness: 316.8940
Cromosoma: 11111100001111100101, Fitness: 53.7194
Cromosoma: 01100111110011001111, Fitness: -438.8498
Cromosoma: 10110010010011100111, Fitness: -232.1273


## Crear la población inicial

In [7]:
def crear_poblacion(tamano):
    return [generar_cromosoma() for _ in range(tamano)]

def resumir_poblacion(poblacion, n=5):
    resumen = []
    for cromosoma in poblacion[:n]:
        x, y = decodificar(cromosoma)
        resumen.append((cromosoma, round(x, 3), round(y, 3), round(fitness(cromosoma), 3)))
    return resumen

poblacion = crear_poblacion(TAM_POBLACION)
resumir_poblacion(poblacion)

[('01111100000000101100', -1.455, -29.29, -1141.076),
 ('11100111110110000100', 25.088, -8.106, 751.608),
 ('10000010001011110011', 0.023, 14.496, -280.163),
 ('11100011100010010110', 24.041, -22.762, 79.79),
 ('10100010011001110111', 7.968, 6.859, 21.914)]

## Selección

Para mantenerlo simple, tomaremos los mejores individuos según su fitness.

In [9]:
def seleccionar_padres(poblacion, cantidad):
    return sorted(poblacion, key=fitness, reverse=True)[:cantidad]

print('\nPadres seleccionados:')
padres = seleccionar_padres(poblacion, 6)
resumir_poblacion(padres, n=len(padres))


Padres seleccionados:


[('00000101111000001010', -30.584, 0.147, 1247.112),
 ('11111000111001110000', 29.276, 6.428, 1087.657),
 ('11100111110110000100', 25.088, -8.106, 751.608),
 ('00100000010110001011', -24.056, -7.674, 693.04),
 ('11101011100011101101', 26.012, -17.405, 498.249),
 ('11110010101101110000', 27.736, 22.194, 368.981)]

## Cruce y mutación

El cruce mezcla dos padres para crear hijos. La mutación cambia algunos bits para mantener diversidad genética.

In [10]:
def cruce(padre1, padre2):
    if random.random() > PROB_CRUCE:
        return padre1, padre2

    punto = random.randint(1, LONGITUD_CROMOSOMA - 1)
    hijo1 = padre1[:punto] + padre2[punto:]
    hijo2 = padre2[:punto] + padre1[punto:]
    return hijo1, hijo2

def mutar(cromosoma):
    bits = list(cromosoma)
    for i in range(len(bits)):
        if random.random() < PROB_MUTACION:
            bits[i] = '1' if bits[i] == '0' else '0'
    return ''.join(bits)

hijo1, hijo2 = cruce(padres[0], padres[1])
print('Padre 1:', padres[0])
print('Padre 2:', padres[1])
print('Hijo 1 :', mutar(hijo1))
print('Hijo 2 :', mutar(hijo2))

Padre 1: 00000101111000001010
Padre 2: 11111000111001110000
Hijo 1 : 00100101111001110000
Hijo 2 : 11111010111000001010


## Crear una nueva generación

Conservamos al mejor individuo actual y llenamos el resto de la población con hijos generados a partir de los mejores padres.

In [12]:
def crear_nueva_generacion(poblacion):
    mejor_actual = max(poblacion, key=fitness)
    nueva_poblacion = [mejor_actual]
    padres = seleccionar_padres(poblacion, max(2, TAM_POBLACION // 2))

    while len(nueva_poblacion) < TAM_POBLACION:
        padre1, padre2 = random.sample(padres, 2)
        hijo1, hijo2 = cruce(padre1, padre2)
        nueva_poblacion.append(mutar(hijo1))
        if len(nueva_poblacion) < TAM_POBLACION:
            nueva_poblacion.append(mutar(hijo2))

    return nueva_poblacion

nueva_poblacion = crear_nueva_generacion(poblacion)
print('\nNueva generación (primeros 5 individuos):')
resumir_poblacion(nueva_poblacion)


Nueva generación (primeros 5 individuos):


[('00000101111000001010', -30.584, 0.147, 1247.112),
 ('01101000101101000111', -6.258, 19.669, -463.588),
 ('00011010011010111111', -25.534, 11.293, 699.245),
 ('01011010011001100010', -9.768, 5.566, 85.92),
 ('11111000111001000010', 29.276, 3.595, 1125.517)]

## Ejecutar el algoritmo completo

Ahora repetimos selección, cruce y mutación durante varias generaciones y vamos guardando el mejor individuo de cada una.

In [13]:
def ejecutar_algoritmo_genetico():
    poblacion = crear_poblacion(TAM_POBLACION)
    historial = []
    mejor_global = max(poblacion, key=fitness)

    for generacion in range(1, GENERACIONES + 1):
        mejor_generacion = max(poblacion, key=fitness)

        if fitness(mejor_generacion) > fitness(mejor_global):
            mejor_global = mejor_generacion

        x, y = decodificar(mejor_generacion)
        historial.append({
            'generacion': generacion,
            'cromosoma': mejor_generacion,
            'x': x,
            'y': y,
            'z': fitness(mejor_generacion),
        })

        poblacion = crear_nueva_generacion(poblacion)

    return historial, mejor_global

## Revisar el resultado

El máximo continuo de la función en el dominio ocurre en $x=-32$ y $y=0$, con valor teórico $z=1365.3333$. El algoritmo genético buscará una aproximación a ese valor.

In [14]:
historial, mejor = ejecutar_algoritmo_genetico()

for fila in historial[:5]:
    print(
        f"Gen {fila['generacion']:02d}: "
        f"x={fila['x']:.4f}, y={fila['y']:.4f}, z={fila['z']:.4f}"
    )

print('...')
x_mejor, y_mejor = decodificar(mejor)
z_mejor = fitness(mejor)
z_teorico = (4 * (-32)**2 - 4 * 0**2) / 3

print(f'Mejor cromosoma: {mejor}')
print(f'x = {x_mejor:.4f}')
print(f'y = {y_mejor:.4f}')
print(f'z = {z_mejor:.4f}')
print(f'Óptimo teórico continuo = {z_teorico:.4f}')
print(f'Error absoluto = {abs(z_teorico - z_mejor):.4f}')

Gen 01: x=30.7537, y=2.4868, z=1252.8050
Gen 02: x=30.7537, y=2.4868, z=1252.8050
Gen 03: x=-31.3226, y=2.4868, z=1299.8932
Gen 04: x=-31.3226, y=0.5777, z=1307.6937
Gen 05: x=-31.8152, y=-0.0997, z=1349.6002
...
Mejor cromosoma: 00000000001000001000
x = -32.0000
y = 0.0235
z = 1365.3326
Óptimo teórico continuo = 1365.3333
Error absoluto = 0.0007
